<a href="https://colab.research.google.com/github/Dani2003/paper-implementations/blob/main/Implementation_of_DISCRETE_DIFFUSION_MODELS_WITH_MLLMS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
# [CELL 1 - Environment Setup]
!pip install -q torch torchvision transformers datasets matplotlib

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer
import numpy as np
import matplotlib.pyplot as plt
import math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print("MeDiM (Medical Discrete Diffusion Model) environment initialized.")

Using device: cuda
MeDiM (Medical Discrete Diffusion Model) environment initialized.


In [30]:
# [CELL 2 - Discrete Tokenizers & Synthetic Dataset]
from transformers import GPT2TokenizerFast

text_tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
text_tokenizer.pad_token = text_tokenizer.eos_token
TEXT_VOCAB_SIZE = len(text_tokenizer)

VISUAL_VOCAB_SIZE = 1024
NUM_VISUAL_TOKENS = 16

VISUAL_OFFSET = 0
TEXT_OFFSET = VISUAL_VOCAB_SIZE
MASK_TOKEN_ID = VISUAL_VOCAB_SIZE + TEXT_VOCAB_SIZE
PAD_TOKEN_ID  = MASK_TOKEN_ID + 1
TOTAL_VOCAB_SIZE = PAD_TOKEN_ID + 1

synthetic_data = [
    {"image_tokens": [10, 45, 88, 102, 12, 44, 89, 101, 15, 46, 90, 103, 11, 47, 91, 104],
     "report": "Normal cardiothoracic ratio. Lungs are clear without focal consolidation."},
    {"image_tokens": [500, 512, 530, 600, 501, 513, 531, 601, 502, 514, 532, 602, 503, 515, 533, 603],
     "report": "Bilateral pleural effusion present. Moderate cardiomegaly noted."},
] * 50

def prepare_multimodal_sequence(item):
    vis_seq = [t + VISUAL_OFFSET for t in item["image_tokens"]]
    text_ids = text_tokenizer.encode(item["report"], truncation=True, max_length=32)
    text_seq = [t + TEXT_OFFSET for t in text_ids]
    return torch.tensor(vis_seq + text_seq, dtype=torch.long)

dataset = [prepare_multimodal_sequence(item) for item in synthetic_data]
print(f"Dataset ready: {len(dataset)} samples | Unified Vocab Size: {TOTAL_VOCAB_SIZE}")

Dataset ready: 100 samples | Unified Vocab Size: 51283


In [31]:
# [CELL 3 - MeDiM Model Architecture]

class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, x):
        device = x.device
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = x[:, None] * emb[None, :]
        return torch.cat((emb.sin(), emb.cos()), dim=-1)

class MeDiMBackbone(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=4):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        # Fix: Initialize pos_emb with standard normal weights for fast convergence
        self.pos_emb = nn.Parameter(torch.randn(1, 128, d_model) * 0.02)

        self.time_mlp = nn.Sequential(
            SinusoidalPosEmb(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model)
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            activation="gelu",
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x, t):
        B, L = x.shape
        h = self.token_emb(x) + self.pos_emb[:, :L, :]
        t_emb = self.time_mlp(t).unsqueeze(1)
        h = h + t_emb
        out = self.transformer(h)
        return self.head(out)

model = MeDiMBackbone(vocab_size=TOTAL_VOCAB_SIZE).to(device)
diffusion_engine = DiscreteAbsorbingDiffusion(mask_token_id=MASK_TOKEN_ID)
print("Model initialized with fixed positional embeddings.")

Model initialized with fixed positional embeddings.


In [32]:
# [CELL 4 - Discrete Diffusion Scheduler & Loss]

class DiscreteAbsorbingDiffusion:
    def __init__(self, mask_token_id):
        self.mask_token_id = mask_token_id

    def q_sample(self, x_0, t):
        """
        Forward Noise Process: Corrupt x_0 to x_t by replacing tokens with [MASK]
        Probability of masking gamma(t) = t
        """
        B, L = x_0.shape
        # Mask probability increases linearly with t
        mask_prob = t.unsqueeze(-1).expand(B, L)

        # Random matrix to determine masked positions
        rand = torch.rand((B, L), device=x_0.device)
        mask_flags = rand < mask_prob

        x_t = x_0.clone()
        x_t[mask_flags] = self.mask_token_id
        return x_t, mask_flags

    def compute_loss(self, model, x_0):
        B, L = x_0.shape

        # Sample continuous timestep t ~ U(0.001, 0.999)
        t = torch.rand(B, device=x_0.device) * 0.998 + 0.001

        # Corrupt inputs
        x_t, mask_flags = self.q_sample(x_0, t)

        # Predict logits with MeDiM
        logits = model(x_t, t)

        # Cross Entropy Loss ONLY on the masked token positions
        loss = F.cross_entropy(
            logits[mask_flags],
            x_0[mask_flags]
        )
        return loss

diffusion_engine = DiscreteAbsorbingDiffusion(mask_token_id=MASK_TOKEN_ID)
test_loss = diffusion_engine.compute_loss(model, dummy_x)
print(f"Discrete Diffusion Forward Pass verified. Initial Loss: {test_loss.item():.4f}")

Discrete Diffusion Forward Pass verified. Initial Loss: 10.9557


In [33]:
# [CELL 5 - Stable Training to Full Convergence]
from torch.utils.data import DataLoader

def pad_collate(batch):
    max_len = max(len(s) for s in batch)
    padded = torch.full((len(batch), max_len), PAD_TOKEN_ID, dtype=torch.long)
    for i, s in enumerate(batch):
        padded[i, :len(s)] = s
    return padded

train_loader = DataLoader(dataset, batch_size=16, shuffle=True, collate_fn=pad_collate)
# Fix: Reduced LR from 1e-3 to 3e-4 to eliminate overshooting
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

model.train()
epochs = 60

print("Training MeDiM to full convergence...")
print("=" * 60)
for epoch in range(1, epochs + 1):
    total_loss = 0.0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        # Compute discrete diffusion loss ignoring PAD tokens
        B, L = batch.shape
        t = torch.rand(B, device=batch.device) * 0.98 + 0.01
        x_t, mask_flags = diffusion_engine.q_sample(batch, t)
        mask_flags = mask_flags & (batch != PAD_TOKEN_ID)

        if mask_flags.any():
            logits = model(x_t, t)
            loss = F.cross_entropy(logits[mask_flags], batch[mask_flags])
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch [{epoch:02d}/{epochs:02d}] | Discrete Diffusion Loss: {avg_loss:.4f}")

print("=" * 60)
print("Training Complete.")

Training MeDiM to full convergence...
Epoch [01/60] | Discrete Diffusion Loss: 10.1048
Epoch [10/60] | Discrete Diffusion Loss: 3.9734
Epoch [20/60] | Discrete Diffusion Loss: 3.5068
Epoch [30/60] | Discrete Diffusion Loss: 0.6738
Epoch [40/60] | Discrete Diffusion Loss: 0.2283
Epoch [50/60] | Discrete Diffusion Loss: 0.0547
Epoch [60/60] | Discrete Diffusion Loss: 0.1296
Training Complete.


In [34]:
# [CELL 6 - Generative Sampling Benchmark]

@torch.no_grad()
def generate_medim(model, prompt_seq=None, target_len=33, num_steps=10):
    model.eval()

    x = torch.full((1, target_len), MASK_TOKEN_ID, dtype=torch.long, device=device)

    if prompt_seq is not None:
        p_len = len(prompt_seq)
        x[0, :p_len] = prompt_seq.to(device)
    else:
        p_len = 0

    text_indices = list(range(p_len, target_len))
    tokens_per_step = math.ceil(len(text_indices) / num_steps)
    timesteps = torch.linspace(1.0, 0.0, num_steps + 1, device=device)

    for i in range(num_steps):
        t_curr = timesteps[i].unsqueeze(0)
        logits = model(x, t_curr)

        text_logits = logits[:, :, TEXT_OFFSET:MASK_TOKEN_ID]
        probs = F.softmax(text_logits, dim=-1)
        max_probs, pred_ids = torch.max(probs, dim=-1)
        pred_text_tokens = pred_ids + TEXT_OFFSET

        currently_masked = [pos for pos in text_indices if x[0, pos] == MASK_TOKEN_ID]
        if not currently_masked:
            break

        confidences = [(pos, max_probs[0, pos].item()) for pos in currently_masked]
        confidences.sort(key=lambda item: item[1], reverse=True)

        for pos, _ in confidences[:tokens_per_step]:
            x[0, pos] = pred_text_tokens[0, pos]

    return x[0]

# Test Case 1: Effusion Image Tokens (500 Series)
sample_img_tokens_1 = torch.tensor([500, 512, 530, 600, 501, 513, 531, 601, 502, 514, 532, 602, 503, 515, 533, 603])
gen_seq_1 = generate_medim(model, prompt_seq=sample_img_tokens_1, target_len=33)
gen_text_ids_1 = [t.item() - TEXT_OFFSET for t in gen_seq_1[NUM_VISUAL_TOKENS:] if TEXT_OFFSET <= t.item() < MASK_TOKEN_ID]

# Test Case 2: Normal Image Tokens (10 Series)
sample_img_tokens_2 = torch.tensor([10, 45, 88, 102, 12, 44, 89, 101, 15, 46, 90, 103, 11, 47, 91, 104])
gen_seq_2 = generate_medim(model, prompt_seq=sample_img_tokens_2, target_len=33)
gen_text_ids_2 = [t.item() - TEXT_OFFSET for t in gen_seq_2[NUM_VISUAL_TOKENS:] if TEXT_OFFSET <= t.item() < MASK_TOKEN_ID]

print("\n" + "="*60)
print("MeDiM GENERATIVE BENCHMARK EVALUATION")
print("="*60)
print(f"Condition 1 (Image 500s) -> Report : {text_tokenizer.decode(gen_text_ids_1).strip()}")
print(f"Condition 2 (Image 10s)  -> Report : {text_tokenizer.decode(gen_text_ids_2).strip()}")
print("="*60)


MeDiM GENERATIVE BENCHMARK EVALUATION
Condition 1 (Image 500s) -> Report : Bilateral pleural effusion present. Moderate cardiomegaly noted. Moderate
Condition 2 (Image 10s)  -> Report : Normal cardiothoracic ratio. Lungs are clear without focal consolidation.
